# LAB3 — Bài 5: Gom cụm MNIST bằng k-Means

**Sinh viên:** Trần Chí Nguyên — **MSSV:** 6451071052.

> Mỗi câu lập trình in họ tên và MSSV theo template. Chạy lần lượt từ trên xuống.

In [1]:
%matplotlib inline
HO_TEN = 'Trần Chí Nguyên'
MSSV = '6451071052'
def in_danh_tinh(): print(f'{HO_TEN} - {MSSV}')


## Câu 5. Gom cụm MNIST bằng k-Means

Dùng ảnh train MNIST và nhãn thật để so sánh. KMeans trên toàn bộ 60.000 ảnh 784 chiều cần nhiều tài nguyên; SAMPLE_SIZE=10000 là cấu hình chạy thực dụng. Đặt None để dùng toàn bộ train set.

In [2]:
in_danh_tinh()
import numpy as np, matplotlib.pyplot as plt, seaborn as sns
from tensorflow.keras.datasets import mnist
from sklearn.cluster import KMeans
from sklearn.metrics import confusion_matrix, classification_report, adjusted_rand_score, normalized_mutual_info_score, silhouette_score
from scipy.optimize import linear_sum_assignment
(train_images,train_labels),_=mnist.load_data()
SAMPLE_SIZE=10000
rng=np.random.default_rng(42)
idx=rng.choice(len(train_images),size=SAMPLE_SIZE,replace=False) if SAMPLE_SIZE is not None else np.arange(len(train_images))
X=train_images[idx].reshape(len(idx),-1).astype('float32')/255.0
y=train_labels[idx]
print('Số mẫu dùng:',len(y),'| shape:',X.shape)
# Elbow khảo sát một tập con để giảm thời gian tính toán
elbow_n=min(1000,len(X)); X_elbow=X[:elbow_n]
inertias=[]
for k in range(2,11):
    km=KMeans(n_clusters=k,random_state=42,n_init=2,max_iter=100); km.fit(X_elbow); inertias.append(km.inertia_)
plt.plot(range(2,11),inertias,marker='o'); plt.xlabel('k'); plt.ylabel('Inertia'); plt.title('MNIST — Elbow'); plt.grid(); plt.show()
# Chữ số có 10 nhãn; dùng k=10 để so sánh trực tiếp
km=KMeans(n_clusters=10,random_state=42,n_init=10)
cluster=km.fit_predict(X)
print('ARI:',adjusted_rand_score(y,cluster),'NMI:',normalized_mutual_info_score(y,cluster))
# Ánh xạ cụm vào nhãn chữ số theo tối đa hóa số khớp
cont=confusion_matrix(y,cluster,labels=np.arange(10))
rows,cols=linear_sum_assignment(-cont)
mapping={c:r for r,c in zip(rows,cols)}
aligned=np.array([mapping[c] for c in cluster])
print('Ánh xạ cụm -> chữ số:',mapping)
print(classification_report(y,aligned,labels=np.arange(10),digits=4,zero_division=0))
sns.heatmap(confusion_matrix(y,aligned,labels=np.arange(10)),cmap='Blues')
plt.xlabel('Dự đoán sau ánh xạ'); plt.ylabel('Nhãn thật'); plt.title('MNIST KMeans — Confusion matrix'); plt.show()
# Hiển thị ảnh và đối chiếu nhãn thật với nhãn dự đoán sau ánh xạ
fig,axes=plt.subplots(1,8,figsize=(14,3))
for j,ax in enumerate(axes): ax.imshow(train_images[idx[j]],cmap='gray'); ax.set_title(f'true {y[j]}\ncluster →{aligned[j]}'); ax.axis('off')
plt.tight_layout(); plt.show()

Trần Chí Nguyên - 6451071052


Số mẫu dùng: 10000 | shape: (10000, 784)


  plt.plot(range(2,11),inertias,marker='o'); plt.xlabel('k'); plt.ylabel('Inertia'); plt.title('MNIST — Elbow'); plt.grid(); plt.show()


ARI: 0.3736239171470736 NMI: 0.4907072686106317
Ánh xạ cụm -> chữ số: {np.int64(2): np.int64(0), np.int64(4): np.int64(1), np.int64(7): np.int64(2), np.int64(5): np.int64(3), np.int64(8): np.int64(4), np.int64(1): np.int64(5), np.int64(0): np.int64(6), np.int64(6): np.int64(7), np.int64(9): np.int64(8), np.int64(3): np.int64(9)}
              precision    recall  f1-score   support

           0     0.9319    0.8096    0.8665       998
           1     0.7148    0.5460    0.6191      1088
           2     0.8800    0.7047    0.7826      1009
           3     0.5149    0.6250    0.5646      1024
           4     0.3980    0.4066    0.4023       964
           5     0.1079    0.1032    0.1055       862
           6     0.8630    0.8096    0.8354      1019
           7     0.3597    0.3869    0.3728      1021
           8     0.4632    0.5615    0.5076      1008
           9     0.3391    0.3923    0.3637      1007

    accuracy                         0.5415     10000
   macro avg     0.

  plt.tight_layout(); plt.show()


## Đánh giá

Dùng ARI/NMI làm thước đo không phụ thuộc nhãn cụm; accuracy/precision/recall/F1 sau Hungarian là cách so sánh dễ đọc. K-Means trên pixel ảnh thô không khai thác cấu trúc không gian nên một số chữ số có thể bị lẫn.